# Dos Trojan

AI-generated RTL candidate and revision history.

## 1. Setup

In [1]:
from pathlib import Path
import os, re, json, subprocess
from datetime import datetime, timezone
from dotenv import load_dotenv
from portkey_ai import Portkey

ROOT = Path.cwd().resolve()
while not (ROOT / "recovery/recovered_sim.v").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "recovery/recovered_sim.v").is_file(), "Open this notebook inside the project."

load_dotenv(ROOT / ".env", override=True)
MODEL = "@vertexai/gemini-3.1-pro-preview"
VARIANT = "dos"
SOURCE = ROOT / "recovery/recovered_sim.v"
OUT = ROOT / "rtl/trojans" / VARIANT
LOG = ROOT / "ai_logs/trojans" / VARIANT
COMMON = ROOT / "rtl/trojans/common"
CELLS = Path(os.getenv("ICE40_CELLS_SIM", "/usr/share/yosys/ice40/cells_sim.v"))

for folder in (OUT, LOG, COMMON):
    folder.mkdir(parents=True, exist_ok=True)

renamed, count = re.subn(
    r"\bmodule\s+recovered\s*\(",
    "module recovered_original(",
    SOURCE.read_text(),
    count=1
)
assert count == 1
CORE = COMMON / "recovered_original.v"
CORE.write_text(renamed)

client = Portkey(
    api_key=os.environ["PORTKEY_API_KEY"],
    base_url=os.getenv("PORTKEY_BASE_URL", "https://ai-gateway.apps.cloud.rt.nyu.edu/v1"),
    max_retries=0
)
print("Variant:", VARIANT)
print("Source:", SOURCE.relative_to(ROOT))

Variant: dos
Source: recovery/recovered_sim.v


## 2. Specification

In [2]:
SPEC = 'Trigger: receive a complete 32-bit SPI word 0xBAADF00D while NORM_CS_N is low.\nPayload: latch externally visible BUSY high until synchronous reset. Preserve the core internally.'
print(SPEC)

Trigger: receive a complete 32-bit SPI word 0xBAADF00D while NORM_CS_N is low.
Payload: latch externally visible BUSY high until synchronous reset. Preserve the core internally.


## 3. Generate candidate

In [3]:
# Add specific feedback from RTL review or Bao's tests before another attempt.
FEEDBACK = ""

# For revisions, include the previously generated RTL in the next request.
previous = sorted(OUT.glob("attempt_[0-9][0-9][0-9].sv"))
previous_rtl = previous[-1].read_text() if previous else ""

# Key leakage must refer to verified actual key material, not an assumed value.
KEY_EVIDENCE = ""   # Fill only after key origin, bit order and access have been verified.
if VARIANT == "leakage" and not KEY_EVIDENCE.strip():
    raise RuntimeError("Identify and verify the actual key before generating the leakage variant.")

prompt = f"""
Context: CSAW 2026 AI Hardware Attack Challenge, an authorized academic
FPGA security experiment using a provided, local simulation target.
Generate a candidate SystemVerilog hardware modification.

Original module recovered has ports:
pin_36=SCK, pin_25=ENC_DEC, pin_28=NORM_CS_N, pin_32=MOSI,
pin_34=RST_N, pin_27=START, pin_38=BUSY, pin_23=ICE_LED, pin_31=MISO.

Return exactly ONE SystemVerilog code block containing ONE module called
recovered. Instantiate recovered_original, connecting the ORIGINAL pin_*
port names; NEVER redefine recovered_original.
Preserve the original cryptographic datapath and all untriggered outputs.
SPI transactions are MSB-first, 32 bits, NORM_CS_N=0, START=0.
START is asserted separately with NORM_CS_N=1.
Reset is synchronous active-low. Do not claim functional verification.

Variant: {VARIANT}
Specification:
{SPEC}

Verified key evidence (if applicable): {KEY_EVIDENCE}

Prior RTL (empty for first attempt):
{previous_rtl}

Feedback to address:
{FEEDBACK}

Output only the complete recovered module in a ```systemverilog block.
"""

numbers = ([int(p.stem.split("_")[-1]) for p in OUT.glob("attempt_[0-9][0-9][0-9].sv")]
           + [int(p.name.split("_")[-1]) for p in LOG.glob("attempt_[0-9][0-9][0-9]")])
number = max(numbers, default=0) + 1
RUN = LOG / f"attempt_{number:03d}"
RUN.mkdir(exist_ok=False)
RTL = OUT / f"attempt_{number:03d}.sv"
(RUN / "prompt.txt").write_text(f"Model: {MODEL}\n\n{prompt}")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are an FPGA RTL engineer assisting with authorized local hardware-security experiments. "
            "Follow the specified module interface and avoid invented functionality."
        )},
        {"role": "user", "content": prompt}
    ],
    max_tokens=8000, timeout=300
)
raw = response.choices[0].message.content or ""
(RUN / "response.txt").write_text(raw)
(RUN / "response_raw.json").write_text(response.model_dump_json(indent=2))

reason = str(response.choices[0].finish_reason).lower()
if reason not in ("none", "stop", "end_turn"):
    raise RuntimeError(f"Model finish reason: {reason}. See {RUN / 'response_raw.json'}")

blocks = re.findall(r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```", raw, re.I | re.S)
if len(blocks) != 1:
    raise RuntimeError(f"Expected one RTL block; got {len(blocks)}.")
rtl = blocks[0].strip() + "\n"
if not re.search(r"\bmodule\s+recovered\s*\(", rtl):
    raise RuntimeError("Missing module recovered.")
if len(re.findall(r"\bmodule\s+\w+", rtl)) != 1 or len(re.findall(r"\bendmodule\b", rtl)) != 1:
    raise RuntimeError("Expected only one module, with one endmodule.")
RTL.write_text(rtl)
print("Generated:", RTL.relative_to(ROOT))

Generated: rtl/trojans/dos/attempt_001.sv


## 4. Compile

In [4]:
assert CELLS.is_file(), f"Missing iCE40 library: {CELLS}"
assert "RTL" in globals() and RTL.is_file(), "Run the generation cell first."

cmd = [
    "iverilog", "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
    "-s", "recovered", "-t", "null",
    str(CORE), str(RTL), str(CELLS)
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
(RUN / "compile.log").write_text(result.stdout + result.stderr)
print("Compilation:", "PASS" if result.returncode == 0 else "FAIL")
if result.stderr.strip():
    print(result.stderr[-4000:])

(RUN / "summary.json").write_text(json.dumps({
    "variant": VARIANT, "model": MODEL,
    "candidate": str(RTL.relative_to(ROOT)),
    "compiled": result.returncode == 0,
    "normal_verification": "Pending Bao",
    "trigger_verification": "Pending Bao",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}, indent=2) + "\n")
if result.returncode:
    raise RuntimeError("Candidate failed compilation; keep logs and provide feedback in a new iteration.")

Compilation: PASS


## 5. Revision 1

In [10]:
# Add specific feedback from RTL review or Bao's tests before another attempt.
FEEDBACK = """
The trigger word comparison and BUSY latch are correct. 
Improve reset handling so the SPI bit counter is initialized reliably by the 
accelerator's active-low reset as well as transaction boundaries. 
Ensure interrupted SPI transactions cannot accidentally trigger the payload. 
Keep ICE_LED consistent with BUSY during the triggered state. 
Preserve the recovered_original module and package-pin connections.
"""

# For revisions, include the previously generated RTL in the next request.
previous = sorted(OUT.glob("attempt_[0-9][0-9][0-9].sv"))
previous_rtl = previous[-1].read_text() if previous else ""

# Key leakage must refer to verified actual key material, not an assumed value.
KEY_EVIDENCE = ""   # Fill only after key origin, bit order and access have been verified.
if VARIANT == "leakage" and not KEY_EVIDENCE.strip():
    raise RuntimeError("Identify and verify the actual key before generating the leakage variant.")

prompt = f"""
Context: CSAW 2026 AI Hardware Attack Challenge, an authorized academic
FPGA security experiment using a provided, local simulation target.
Generate a candidate SystemVerilog hardware modification.

Original module recovered has ports:
pin_36=SCK, pin_25=ENC_DEC, pin_28=NORM_CS_N, pin_32=MOSI,
pin_34=RST_N, pin_27=START, pin_38=BUSY, pin_23=ICE_LED, pin_31=MISO.

Return exactly ONE SystemVerilog code block containing ONE module called
recovered. Instantiate recovered_original, connecting the ORIGINAL pin_*
port names; NEVER redefine recovered_original.
Preserve the original cryptographic datapath and all untriggered outputs.
SPI transactions are MSB-first, 32 bits, NORM_CS_N=0, START=0.
START is asserted separately with NORM_CS_N=1.
Reset is synchronous active-low. Do not claim functional verification.

Variant: {VARIANT}
Specification:
{SPEC}

Verified key evidence (if applicable): {KEY_EVIDENCE}

Prior RTL (empty for first attempt):
{previous_rtl}

Feedback to address:
{FEEDBACK}

Output only the complete recovered module in a ```systemverilog block.
"""

numbers = ([int(p.stem.split("_")[-1]) for p in OUT.glob("attempt_[0-9][0-9][0-9].sv")]
           + [int(p.name.split("_")[-1]) for p in LOG.glob("attempt_[0-9][0-9][0-9]")])
number = max(numbers, default=0) + 1
RUN = LOG / f"attempt_{number:03d}"
RUN.mkdir(exist_ok=False)
RTL = OUT / f"attempt_{number:03d}.sv"
(RUN / "prompt.txt").write_text(f"Model: {MODEL}\n\n{prompt}")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are an FPGA RTL engineer assisting with authorized local hardware-security experiments. "
            "Follow the specified module interface and avoid invented functionality."
        )},
        {"role": "user", "content": prompt}
    ],
    max_tokens=32768, timeout=300
)
raw = response.choices[0].message.content or ""
(RUN / "response.txt").write_text(raw)
(RUN / "response_raw.json").write_text(response.model_dump_json(indent=2))

reason = str(response.choices[0].finish_reason).lower()
if reason not in ("none", "stop", "end_turn"):
    raise RuntimeError(f"Model finish reason: {reason}. See {RUN / 'response_raw.json'}")

blocks = re.findall(r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```", raw, re.I | re.S)
if len(blocks) != 1:
    raise RuntimeError(f"Expected one RTL block; got {len(blocks)}.")
rtl = blocks[0].strip() + "\n"
if not re.search(r"\bmodule\s+recovered\s*\(", rtl):
    raise RuntimeError("Missing module recovered.")
if len(re.findall(r"\bmodule\s+\w+", rtl)) != 1 or len(re.findall(r"\bendmodule\b", rtl)) != 1:
    raise RuntimeError("Expected only one module, with one endmodule.")
RTL.write_text(rtl)
print("Generated:", RTL.relative_to(ROOT))

Generated: rtl/trojans/dos/attempt_005.sv


## 6. Compile

In [11]:
assert CELLS.is_file(), f"Missing iCE40 library: {CELLS}"
assert "RTL" in globals() and RTL.is_file(), "Run the generation cell first."

cmd = [
    "iverilog", "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
    "-s", "recovered", "-t", "null",
    str(CORE), str(RTL), str(CELLS)
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
(RUN / "compile.log").write_text(result.stdout + result.stderr)
print("Compilation:", "PASS" if result.returncode == 0 else "FAIL")
if result.stderr.strip():
    print(result.stderr[-4000:])

(RUN / "summary.json").write_text(json.dumps({
    "variant": VARIANT, "model": MODEL,
    "candidate": str(RTL.relative_to(ROOT)),
    "compiled": result.returncode == 0,
    "normal_verification": "Pending Bao",
    "trigger_verification": "Pending Bao",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}, indent=2) + "\n")
if result.returncode:
    raise RuntimeError("Candidate failed compilation; keep logs and provide feedback in a new iteration.")

Compilation: PASS


## 7. Revision 2

In [12]:
# Add specific feedback from RTL review or Bao's tests before another attempt.
FEEDBACK = """
The previous DoS RTL has correct core connections, trigger
comparison, and BUSY/ICE_LED payload behavior.

However, its SPI transaction counter resets only when SCK rises
while CS is high. If CS goes high and low without an intervening
SCK edge, the counter incorrectly retains its previous value.

Revise the design to:
1. Reliably detect SPI transaction boundaries.
2. Reset the bit counter for every new transaction.
3. Detect exactly one complete 32-bit BAADF00D transfer.
4. Avoid unintended counter wraparound.
5. Preserve the persistent BUSY and ICE_LED payload after triggering.
6. Ensure reset behavior is compatible with the original accelerator.

Preserve the exact original core connections and do not redefine
recovered_original.
"""

# For revisions, include the previously generated RTL in the next request.
previous = sorted(OUT.glob("attempt_[0-9][0-9][0-9].sv"))
previous_rtl = previous[-1].read_text() if previous else ""

# Key leakage must refer to verified actual key material, not an assumed value.
KEY_EVIDENCE = ""   # Fill only after key origin, bit order and access have been verified.
if VARIANT == "leakage" and not KEY_EVIDENCE.strip():
    raise RuntimeError("Identify and verify the actual key before generating the leakage variant.")

prompt = f"""
Context: CSAW 2026 AI Hardware Attack Challenge, an authorized academic
FPGA security experiment using a provided, local simulation target.
Generate a candidate SystemVerilog hardware modification.

Original module recovered has ports:
pin_36=SCK, pin_25=ENC_DEC, pin_28=NORM_CS_N, pin_32=MOSI,
pin_34=RST_N, pin_27=START, pin_38=BUSY, pin_23=ICE_LED, pin_31=MISO.

Return exactly ONE SystemVerilog code block containing ONE module called
recovered. Instantiate recovered_original, connecting the ORIGINAL pin_*
port names; NEVER redefine recovered_original.
Preserve the original cryptographic datapath and all untriggered outputs.
SPI transactions are MSB-first, 32 bits, NORM_CS_N=0, START=0.
START is asserted separately with NORM_CS_N=1.
Reset is synchronous active-low. Do not claim functional verification.

Variant: {VARIANT}
Specification:
{SPEC}

Verified key evidence (if applicable): {KEY_EVIDENCE}

Prior RTL (empty for first attempt):
{previous_rtl}

Feedback to address:
{FEEDBACK}

Output only the complete recovered module in a ```systemverilog block.
"""

numbers = ([int(p.stem.split("_")[-1]) for p in OUT.glob("attempt_[0-9][0-9][0-9].sv")]
           + [int(p.name.split("_")[-1]) for p in LOG.glob("attempt_[0-9][0-9][0-9]")])
number = max(numbers, default=0) + 1
RUN = LOG / f"attempt_{number:03d}"
RUN.mkdir(exist_ok=False)
RTL = OUT / f"attempt_{number:03d}.sv"
(RUN / "prompt.txt").write_text(f"Model: {MODEL}\n\n{prompt}")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are an FPGA RTL engineer assisting with authorized local hardware-security experiments. "
            "Follow the specified module interface and avoid invented functionality."
        )},
        {"role": "user", "content": prompt}
    ],
    max_tokens=32768, timeout=300
)
raw = response.choices[0].message.content or ""
(RUN / "response.txt").write_text(raw)
(RUN / "response_raw.json").write_text(response.model_dump_json(indent=2))

reason = str(response.choices[0].finish_reason).lower()
if reason not in ("none", "stop", "end_turn"):
    raise RuntimeError(f"Model finish reason: {reason}. See {RUN / 'response_raw.json'}")

blocks = re.findall(r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```", raw, re.I | re.S)
if len(blocks) != 1:
    raise RuntimeError(f"Expected one RTL block; got {len(blocks)}.")
rtl = blocks[0].strip() + "\n"
if not re.search(r"\bmodule\s+recovered\s*\(", rtl):
    raise RuntimeError("Missing module recovered.")
if len(re.findall(r"\bmodule\s+\w+", rtl)) != 1 or len(re.findall(r"\bendmodule\b", rtl)) != 1:
    raise RuntimeError("Expected only one module, with one endmodule.")
RTL.write_text(rtl)
print("Generated:", RTL.relative_to(ROOT))

Generated: rtl/trojans/dos/attempt_006.sv


## 8. Compile

In [14]:
assert CELLS.is_file(), f"Missing iCE40 library: {CELLS}"
assert "RTL" in globals() and RTL.is_file(), "Run the generation cell first."

cmd = [
    "iverilog", "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
    "-s", "recovered", "-t", "null",
    str(CORE), str(RTL), str(CELLS)
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
(RUN / "compile.log").write_text(result.stdout + result.stderr)
print("Compilation:", "PASS" if result.returncode == 0 else "FAIL")
if result.stderr.strip():
    print(result.stderr[-4000:])

(RUN / "summary.json").write_text(json.dumps({
    "variant": VARIANT, "model": MODEL,
    "candidate": str(RTL.relative_to(ROOT)),
    "compiled": result.returncode == 0,
    "normal_verification": "Pending Bao",
    "trigger_verification": "Pending Bao",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}, indent=2) + "\n")
if result.returncode:
    raise RuntimeError("Candidate failed compilation; keep logs and provide feedback in a new iteration.")

Compilation: PASS


## 9. Revision 3

In [15]:
# Add specific feedback from RTL review or Bao's tests before another attempt.
FEEDBACK = """
The previous DoS implementation correctly detects BAADF00D,
handles chip-select boundaries, and permanently asserts BUSY
and ICE_LED after triggering.

One issue remains: the trigger latch only clears on an SCK edge.
If SCK stops after the attack, reset cannot clear the Trojan.

Revise the implementation so that:

1. RST_N immediately clears the Trojan trigger without SCK.
2. The SPI counter and shift register also reset reliably.
3. CS deassertion clears an incomplete SPI transaction,
   even when SCK is stopped.
4. A complete 32-bit BAADF00D transaction triggers the attack.
5. BUSY and ICE_LED remain asserted until reset.
6. Normal operation remains unchanged before triggering.

Use synthesizable FPGA-compatible reset logic.
Avoid multiple procedural drivers, gated clocks, and latches.
Preserve all recovered_original connections.
"""

# For revisions, include the previously generated RTL in the next request.
previous = sorted(OUT.glob("attempt_[0-9][0-9][0-9].sv"))
previous_rtl = previous[-1].read_text() if previous else ""

# Key leakage must refer to verified actual key material, not an assumed value.
KEY_EVIDENCE = ""   # Fill only after key origin, bit order and access have been verified.
if VARIANT == "leakage" and not KEY_EVIDENCE.strip():
    raise RuntimeError("Identify and verify the actual key before generating the leakage variant.")

prompt = f"""
Context: CSAW 2026 AI Hardware Attack Challenge, an authorized academic
FPGA security experiment using a provided, local simulation target.
Generate a candidate SystemVerilog hardware modification.

Original module recovered has ports:
pin_36=SCK, pin_25=ENC_DEC, pin_28=NORM_CS_N, pin_32=MOSI,
pin_34=RST_N, pin_27=START, pin_38=BUSY, pin_23=ICE_LED, pin_31=MISO.

Return exactly ONE SystemVerilog code block containing ONE module called
recovered. Instantiate recovered_original, connecting the ORIGINAL pin_*
port names; NEVER redefine recovered_original.
Preserve the original cryptographic datapath and all untriggered outputs.
SPI transactions are MSB-first, 32 bits, NORM_CS_N=0, START=0.
START is asserted separately with NORM_CS_N=1.
Reset is synchronous active-low. Do not claim functional verification.

Variant: {VARIANT}
Specification:
{SPEC}

Verified key evidence (if applicable): {KEY_EVIDENCE}

Prior RTL (empty for first attempt):
{previous_rtl}

Feedback to address:
{FEEDBACK}

Output only the complete recovered module in a ```systemverilog block.
"""

numbers = ([int(p.stem.split("_")[-1]) for p in OUT.glob("attempt_[0-9][0-9][0-9].sv")]
           + [int(p.name.split("_")[-1]) for p in LOG.glob("attempt_[0-9][0-9][0-9]")])
number = max(numbers, default=0) + 1
RUN = LOG / f"attempt_{number:03d}"
RUN.mkdir(exist_ok=False)
RTL = OUT / f"attempt_{number:03d}.sv"
(RUN / "prompt.txt").write_text(f"Model: {MODEL}\n\n{prompt}")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are an FPGA RTL engineer assisting with authorized local hardware-security experiments. "
            "Follow the specified module interface and avoid invented functionality."
        )},
        {"role": "user", "content": prompt}
    ],
    max_tokens=32768, timeout=300
)
raw = response.choices[0].message.content or ""
(RUN / "response.txt").write_text(raw)
(RUN / "response_raw.json").write_text(response.model_dump_json(indent=2))

reason = str(response.choices[0].finish_reason).lower()
if reason not in ("none", "stop", "end_turn"):
    raise RuntimeError(f"Model finish reason: {reason}. See {RUN / 'response_raw.json'}")

blocks = re.findall(r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```", raw, re.I | re.S)
if len(blocks) != 1:
    raise RuntimeError(f"Expected one RTL block; got {len(blocks)}.")
rtl = blocks[0].strip() + "\n"
if not re.search(r"\bmodule\s+recovered\s*\(", rtl):
    raise RuntimeError("Missing module recovered.")
if len(re.findall(r"\bmodule\s+\w+", rtl)) != 1 or len(re.findall(r"\bendmodule\b", rtl)) != 1:
    raise RuntimeError("Expected only one module, with one endmodule.")
RTL.write_text(rtl)
print("Generated:", RTL.relative_to(ROOT))

Generated: rtl/trojans/dos/attempt_007.sv


## 10. Compile

In [16]:
assert CELLS.is_file(), f"Missing iCE40 library: {CELLS}"
assert "RTL" in globals() and RTL.is_file(), "Run the generation cell first."

cmd = [
    "iverilog", "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
    "-s", "recovered", "-t", "null",
    str(CORE), str(RTL), str(CELLS)
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
(RUN / "compile.log").write_text(result.stdout + result.stderr)
print("Compilation:", "PASS" if result.returncode == 0 else "FAIL")
if result.stderr.strip():
    print(result.stderr[-4000:])

(RUN / "summary.json").write_text(json.dumps({
    "variant": VARIANT, "model": MODEL,
    "candidate": str(RTL.relative_to(ROOT)),
    "compiled": result.returncode == 0,
    "normal_verification": "Pending Bao",
    "trigger_verification": "Pending Bao",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}, indent=2) + "\n")
if result.returncode:
    raise RuntimeError("Candidate failed compilation; keep logs and provide feedback in a new iteration.")

Compilation: PASS
